In [1]:
from pickletools import optimize

from torch.onnx.ops import attention
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

model_name = "distilbert-base-uncased"

tokenizer = DistilBertTokenizer.from_pretrained(model_name)

model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [2]:
import pandas as pd

df_train = pd.read_parquet("../data/processed/train_clean.parquet")
df_test = pd.read_parquet("../data/processed/test_clean.parquet")

In [3]:
sample = tokenizer(df_train["text"][0], truncation=True, max_length=512, padding="max_length", return_tensors="pt")
print(sample.keys())
print(sample["input_ids"].shape)

KeysView({'input_ids': tensor([[  101,  1045, 12524,  1045,  2572,  8025,  6672,  7174,  2860,  2013,
          2026,  2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,
          2009,  2043,  2009,  2001,  2034,  2207,  1999,  3476,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  2149,  8205,
          2065,  2009,  2412,  2699,  2000,  4607,  2023,  2406,  3568,  2108,
          1037,  5470,  1997,  3152,  2641,  6801,  1045,  2428,  2018,  2000,
          2156,  2023,  2005,  2870, 10760,  5436,  2003,  8857,  2105,  1037,
          2402,  4467,  3689,  3076,  2315, 14229,  2040,  4122,  2000,  4553,
          2673,  2016,  2064,  2055,  2166,  1999,  3327,  2016,  4122,  2000,
          3579,  2014,  3086,  2015,  2000,  2437,  2070,  4066,  1997,  4516,
          2006,  2054,  1996,  2779, 25430, 14728,  2245,  2055,  3056,  2576,
          3314,  2107,  2004,  1996,  5148,  2162,  1998,  2679,  3314,  1999,
          1996,  2142,  2163,

In [4]:
from torch.utils.data import Dataset


class IMDbDatasetBERT(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = tokenizer(text, truncation=True, max_length=512, padding="max_length", return_tensors="pt")
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return input_ids, attention_mask, label

In [5]:
train_dataset = IMDbDatasetBERT(df_train["text"].values, df_train["label"].values)
test_dataset = IMDbDatasetBERT(df_test["text"].values, df_test["label"].values)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [11]:
input_ids, attention_mask, labels = next(iter(train_loader))
print(input_ids.shape, attention_mask.shape, labels.shape)

torch.Size([16, 512]) torch.Size([16, 512]) torch.Size([16])


In [12]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

In [14]:
epochs = 3
losses = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for batch_idx, (input_ids, attention_mask, labels) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Batch {batch_idx} | Loss {loss.item():.4f}")

    print(f"Epoch {epoch} | Loss {epoch_loss / len(train_loader):.4f}")

Batch 0 | Loss 0.1919
Batch 50 | Loss 0.3011
Batch 100 | Loss 0.2258
Batch 150 | Loss 0.2800
Batch 200 | Loss 0.5226
Batch 250 | Loss 0.2001
Batch 300 | Loss 0.2627
Batch 350 | Loss 0.2895
Batch 400 | Loss 0.0993
Batch 450 | Loss 0.1002
Batch 500 | Loss 0.1861
Batch 550 | Loss 0.1344
Batch 600 | Loss 0.0426
Batch 650 | Loss 0.2491
Batch 700 | Loss 0.0903
Batch 750 | Loss 0.3306
Batch 800 | Loss 0.4702
Batch 850 | Loss 0.0187
Batch 900 | Loss 0.4638


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=True)

In [ ]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device).long()

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = logits.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")
print(f"Correctly predicted {correct} out of {total} samples")